# 02_05 Token embeddings

*📖 Book: section 2.7*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → [numbers] → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **numbers**
>
> Project files this connects to:
>
> - `src/homer_ai/model.py`
> - `src/homer_ai/tokenizer.py`
>
> **This notebook:** A token ID like 41 is just a label. How does the model get something it can compute with, that can carry meaning?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- Why a token ID is only a label, not something to compute with
- What an embedding is: a list of numbers for each token, learned in training
- What the lookup table looks like, and why its shape is 137 × 32
- What our trained model's embeddings really show (and what they don't)

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **vector** | a list of numbers, used as a point or an arrow | `[9, 1]` |
| **embedding** | a vector that stands for one token; the model learns its numbers during training | 32 numbers for `α` |
| **embedding table** | the lookup table of all embeddings: one row per token | 137 rows |
| **embed_dim** | how many numbers each embedding has: the table's number of columns | 32 |
| **cosine similarity** | how much two vectors point the same way: `1` same, `0` unrelated, `-1` opposite (see [A_03](../A_toolbox/A_03_dot_product.ipynb)) | `[9, 1]` and `[8, 1]` → `1.0` |

## 3. The idea

### ❓ A token ID like 41 is just a label. Why can't the model compute with it?

Because the number was handed out for a boring reason: the tokenizer sorted the characters, and `π` landed in place 41. Nothing about `π` is "41-ish".

```
ID:    38   39   40   41   42   43   44
char:   ν    ξ    ο    π    ρ    ς    σ
```

`ο` (40) isn't "smaller" than `π` (41) in any way that helps, and `ν` (38) isn't "closer" to `ο` than `ξ` is. If the model did maths on 41, it would find patterns in alphabet order, not in Greek.

👉 **Predict first:** which characters have the IDs 38 to 44?

In [ ]:
from homer_ai.tokenizer import CharTokenizer
from homer_ai.train_homer import BOOKS_DIR, load_books

tokenizer = CharTokenizer(load_books(BOOKS_DIR)[0])
for i in range(38, 45):
    print(i, tokenizer.id_to_char[i])

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The IDs are just positions in the sorted list. We need something richer than one number.

</details>

### ❓ What could we use instead?

We describe each character with several numbers, one for each thing we care about. It's like rating films by `[Action, Romance]` in [A_03](../A_toolbox/A_03_dot_product.ipynb). A list of numbers like this is called a **vector**.

**If the dot product is new to you, read [A_03](../A_toolbox/A_03_dot_product.ipynb) now** (about 10 minutes). The rest of this notebook measures how alike two vectors are, and all of chapter 3 is built on it.

Let's try it by hand for four letters, with two features I made up: `[vowel, consonant]`, each from 0 to 10.

| letter | vector | meaning |
|---|---|---|
| `α` | `[9, 1]` | very vowel-like |
| `ά` | `[8, 1]` | `α` with an accent: still a vowel |
| `ν` | `[1, 9]` | very consonant-like |
| `ρ` | `[1, 8]` | a consonant too |

Now similar letters have similar vectors, and we can measure that with the **cosine similarity** from A_03: `1` means the same direction, near `0` means unrelated.

👉 **Predict first:** will `α` and `ά` score high or low? And `α` and `ν`?

In [ ]:
from math import sqrt


def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b, strict=True))
    return dot / (sqrt(sum(x * x for x in a)) * sqrt(sum(y * y for y in b)))


by_hand = {"α": [9, 1], "ά": [8, 1], "ν": [1, 9], "ρ": [1, 8]}
for first, second in [("α", "ά"), ("ν", "ρ"), ("α", "ν"), ("ά", "ρ")]:
    score = cosine(by_hand[first], by_hand[second])
    print(f"{first} {second}  {score:.2f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`α`/`ά` and `ν`/`ρ` score `1.00`, so they point the same way. A vowel against a consonant scores about `0.2`, which is far apart. Similar tokens, similar vectors.

</details>

### ❓ So what is an embedding?

The same idea, with one big change: **we don't choose the numbers, the model learns them.** An embedding is a vector for each token. At the start of training its numbers are random. Then, while the model practises on the flashcards from [02_04](02_04_sliding_window.ipynb), it nudges every embedding, so that tokens which behave alike (and so help predict the same next letters) drift towards similar vectors.

Nobody tells it "vowel" or "consonant". Our hand-made features had 2 numbers. The model uses 32, and no single one of them has a name. That count (32) is called `embed_dim`.

### ❓ Where are all these vectors kept, and what is its shape?

In a lookup table called the **embedding table**, with one row for each token. Have a guess at its shape:

```
rows    = how many different tokens we have   (the vocabulary)
columns = how many numbers each token gets   (embed_dim)
```

👉 **Predict first:** with 137 tokens and `embed_dim=32`, what is the table's shape, and how many weights is that?

In [ ]:
import torch
from torch import nn

table = nn.Embedding(tokenizer.vocab_size, 32)
print("shape:  ", tuple(table.weight.shape))
print("weights:", table.weight.numel(), "=", tokenizer.vocab_size, "×", 32)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`(137, 32)`, which is 4,384 weights. Back in 01_02 that was one of the two biggest parts of our 11,840 weights. [A_04](../A_toolbox/A_04_pytorch_in_this_repo.ipynb) goes through `nn.Embedding` line by line. Here we only need the idea.

</details>

### ❓ What does "look up" really do?

It picks rows, and that's all. Here's a tiny table: 5 tokens, 3 numbers each (`seed` fixes the random start, so you get the same numbers as this page).

👉 **Predict first:** which rows of the table will come back for the IDs `[1, 3]`?

In [ ]:
torch.manual_seed(0)
tiny = nn.Embedding(5, 3)
print(tiny.weight)
print(tiny(torch.tensor([1, 3])))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The two rows it printed are exactly row 1 and row 3 of the table. Looking up an ID just means taking that row.

</details>

### ❓ What does that look like for a real word?

Take `ἄνδρα` ("man", the first word of the *Odyssey*). Five characters, five IDs, five rows.

The text is `ἄ ν δ ρ α`, the IDs are `58 38 29 42 26`, and each ID picks one row of 32 numbers. So the result is a 5 × 32 table.

The model never sees `ἄνδρα`. It sees this 5 × 32 table of numbers.

👉 **Predict first:** what shape will the result have?

In [ ]:
ids = torch.tensor(tokenizer.encode("ἄνδρα"))
print("IDs:  ", ids.tolist())
print("shape:", tuple(table(ids).shape))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`[58, 38, 29, 42, 26]` becomes shape `(5, 32)`: one row per character. (This table is untrained, so its numbers are random. Next we look at the trained one.)

</details>

**Look at:** the arrows. Each ID on the left picks one row of the table and puts it in the stack on the right, in the order of the word. Row 58, for `ἄ`, is orange.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import (
    ACCENT,
    EDGE,
    INK,
    MUTED,
    arrow,
    box,
    diagram_axes,
    studio_figure,
    tag,
)

word = "ἄνδρα"
word_ids = tokenizer.encode(word)
fig = studio_figure("Embedding lookup", "Five IDs pull out five rows", (9, 6.5))
ax = diagram_axes(fig)

tag(ax, 2.0, 5.95, "the embedding table (137 × 32)", INK, 10, weight="bold")
tag(ax, 8.2, 5.95, "what the model reads (5 × 32)", INK, 10, weight="bold")
table_y = {}
for k, token_id in enumerate(sorted(word_ids)):
    y = 5.2 - 0.95 * k
    table_y[token_id] = y
    char = tokenizer.id_to_char[token_id]
    color = ACCENT if token_id == word_ids[0] else EDGE
    box(ax, 2.0, y, 3.0, 0.55, f"{token_id:>3} {char}  [32 numbers]", color, mono=True)
    if k < 4:
        tag(ax, 2.0, y - 0.47, "⋮", MUTED, 10)
for k, token_id in enumerate(word_ids):
    y = 4.9 - 0.8 * k
    color = ACCENT if k == 0 else EDGE
    label = f"{word[k]}  [32 numbers]"
    box(ax, 8.2, y, 3.0, 0.55, label, color, mono=True)
    arrow(ax, (3.55, table_y[token_id]), (6.65, y), color=ACCENT if k == 0 else MUTED)
tag(ax, 5.1, 1.45, "look up each ID: take its row", MUTED, 9)
plt.show()

### ❓ Your turn: can you look up `μοι`?

Same lookup, another word: `μοι` (to me), the second word of the *Odyssey*. Get its rows from `table`.

👉 **Predict first:** what shape will come back?

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
# 👉 do: encode "μοι", make the IDs a tensor, and look them up in `table`
moi_rows = None

In [ ]:
from homer_ai.lessons import check

check(
    "the shape of the rows of μοι",
    moi_rows,
    table(torch.tensor(tokenizer.encode("μοι"))),
    show=lambda rows: tuple(rows.shape),
    coach="Shape (3,) means you still have the 3 IDs: pass them through table(...). "
    "A list instead of a shape means the IDs aren't a tensor yet: wrap them in "
    "torch.tensor(...). And tiny only has 5 rows, so use table.",
)

<details><summary>💡 Hint</summary>

It's the `ἄνδρα` cell with another word: `ids = torch.tensor(tokenizer.encode(...))`, then `table(ids)`.

</details>

<details><summary>✅ Answer</summary>

```python
moi_rows = table(torch.tensor(tokenizer.encode("μοι")))
```

Shape `(3, 32)`: three characters, so three rows of 32 numbers. A longer text just means more rows. The 32 never changes.

</details>

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Did our trained model really give similar letters similar rows?

Let's measure it and not guess. We load the saved model (`homer_model.pt`; we only look, we don't change it) and take the cosine similarity between rows. Some pairs to try:

- `α` and its accented forms `ά`, `ὰ`, `ᾶ` (the same letter, with different accent marks)
- `α` and the consonants `ν`, `ρ`
- two vowels, `ε` and `ο`

👉 **Predict first:** which pair do you think scores highest?

In [ ]:
import torch.nn.functional as F

from homer_ai.lessons import load_homer_model

trained_tokenizer, trained_model = load_homer_model()  # read-only
have_model = trained_model is not None
if have_model:
    learned = trained_model.embedding.weight.detach()  # the learned table
    print("trained table shape:", tuple(learned.shape))

    def similarity(a, b):
        row_a = learned[trained_tokenizer.encode(a)[0]]
        row_b = learned[trained_tokenizer.encode(b)[0]]
        return F.cosine_similarity(row_a, row_b, dim=0).item()

    pairs = [("α", "ά"), ("α", "ᾶ"), ("α", "ὰ"), ("α", "ν"), ("α", "ρ"), ("ε", "ο")]
    for first, second in [*pairs, ("α", "ε")]:
        print(f"{first} {second}  {similarity(first, second):+.2f}")
else:
    print("no trained model yet: make one with `uv run homer-train`")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Small numbers, mostly. `α` with `ά` and `ᾶ` is about `+0.25`, and with `ὰ` only `+0.05`. `α` with the consonants `ν` and `ρ` is about `-0.05`. And `α` with the vowel `ε` is the highest, at `+0.40`. So there's a hint of "vowels together, consonants apart", but it's weak. A picture with more pairs comes next.

</details>

### ❓ Can I see all the pairs at once?

Yes, with a heatmap. Each square is the cosine similarity of two characters. Green means the rows point the same way, pink-red means the opposite way, and dark means unrelated. The diagonal is each character against itself, so it's always `1`.

👉 **Predict first:** will the block of vowels (`α ά ὰ ᾶ ε ο ι`) look clearly greener than the vowel-and-consonant squares?

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import GOOD_BAD, INK, MUTED, studio_figure, text_on

if have_model:
    chars = list("αάὰᾶεοινρς")
    rows = learned[[trained_tokenizer.encode(ch)[0] for ch in chars]]
    unit = F.normalize(rows, dim=1)  # each row scaled to length 1
    sims = (unit @ unit.T).detach().numpy()  # all the cosine similarities at once

    fig = studio_figure("Trained embeddings", "How alike are two characters' rows?")
    ax = fig.add_axes((0.13, 0.08, 0.72, 0.68))
    ax.set_facecolor("none")
    ax.imshow(sims, cmap=GOOD_BAD, vmin=-1, vmax=1)
    ax.set_xticks(range(len(chars)), chars, fontsize=14, color=INK)
    ax.set_yticks(range(len(chars)), chars, fontsize=14, color=INK)
    ax.xaxis.tick_top()
    ax.tick_params(length=0)
    for side in ax.spines.values():
        side.set_visible(False)
    for r in range(len(chars)):
        for c in range(len(chars)):
            value = sims[r, c]
            ax.text(c, r, f"{value:.2f}", ha="center", va="center", fontsize=8,
                    color=text_on(value, 1))
    note = "green = alike, pink-red = opposite, dark = unrelated"
    fig.text(0.13, 0.03, note, color=MUTED, fontsize=9)
    plt.show()
else:
    print("no trained model yet: make one with `uv run homer-train`")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

A mix. The vowels aren't one clean green block, and a pair of consonants can be just as green as the vowel squares (`ν`/`ς` is `+0.34`). The strongest pair is `α`/`ο` at `+0.53`. Most squares are dark, meaning unrelated.

</details>

### ❓ Is that a failure? What would we expect?

Not quite, but let's look at what the numbers really say. With 32 random numbers, two unrelated rows usually score about `±0.14` just by chance. Most of our scores sit inside that noise, so what they show is at most a faint hint of letter families. It's not a clear map.

Why? Our model is tiny (11,840 weights, one attention layer), it practised on a small amount of text, and rare letters like `ὰ` appear so few times that their rows barely moved from the random start. Real models, with far more text and far more weights, make much clearer maps.

👉 **Predict first:** what is the typical size of the cosine similarity between two *random* 32-number rows?

In [ ]:
torch.manual_seed(1)
random_rows = F.normalize(torch.randn(2000, 32), dim=1)
random_sims = random_rows @ random_rows.T
off_diagonal = random_sims[~torch.eye(2000, dtype=torch.bool)]
print(f"typical size of a random pair: {off_diagonal.abs().mean():.2f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

About `0.14`. So `α`/`ε` at `+0.40` stands out a little, while `+0.05` or `-0.06` is just noise. The thing to take away is the idea: embeddings are numbers the model learns, so that similar tokens *can* end up similar. How well that works depends on the data and on the model.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Does `α` get a different row each time it appears?

No. It's one row per token, the same wherever `α` sits. And that's a problem, because then the model can't tell `ἄνδρα` from the same letters in another order. [02_06](02_06_positions.ipynb) deals with that.

### ❓ Can I choose `embed_dim`?

Yes, it's a setting. Bigger means a richer description of each token, but also more weights to learn. We use 32. Big models use thousands.

## 5. Recap

- A token ID is only a label; an embedding gives each token a vector the model can compute with.
- The embedding table has one row per token: 137 × 32 = 4,384 weights, and lookup just picks rows.
- The numbers are learned; in our tiny model they show only a faint hint of similar letters being alike.

**You can now:**

- make an embedding table with `nn.Embedding` and say how many weights it has
- look up the rows of a word and predict the shape you get
- compare two rows with cosine similarity

**Next:** [02_06: Positions](02_06_positions.ipynb): the same letter gets the same row everywhere, so how does the model know where it is?